In [ ]:
import torch
import torchvision
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
import math
from pathlib import Path
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import pandas as pd
import random
import numpy as np
import timm
from tqdm.auto import tqdm
import copy
import gc
from torch.amp import autocast, GradScaler
from collections import OrderedDict


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

In [ ]:
class NICOPPDataset(Dataset):
    def __init__(
        self,
        root_path,
        domain_groups,
        splits=("train",),
        real_domains=None,
        class_to_idx=None,
        transform=None
    ):
        self.root_path = Path(root_path)
        self.domain_groups = domain_groups if isinstance(domain_groups, (list, tuple)) else [domain_groups]
        self.splits = splits if isinstance(splits, (list, tuple)) else [splits]
        self.real_domains = set(real_domains) if real_domains is not None else None
        self.transform = transform
        self.samples = []

        # Build class mapping
        if class_to_idx is None:
            class_names = set()

            for domain_group in self.domain_groups:
                for split in self.splits:
                    split_path = self.root_path / domain_group / split
                    if not split_path.exists():
                        continue

                    for real_domain_dir in split_path.iterdir():
                        if not real_domain_dir.is_dir():
                            continue

                        if self.real_domains is not None and real_domain_dir.name not in self.real_domains:
                            continue

                        for class_dir in real_domain_dir.iterdir():
                            if class_dir.is_dir():
                                class_names.add(class_dir.name)

            self.class_to_idx = {
                class_name: idx
                for idx, class_name in enumerate(sorted(class_names))
            }
        else:
            self.class_to_idx = class_to_idx

        # Collect samples
        for domain_group in self.domain_groups:
            for split in self.splits:
                split_path = self.root_path / domain_group / split
                if not split_path.exists():
                    continue

                for real_domain_dir in split_path.iterdir():
                    if not real_domain_dir.is_dir():
                        continue

                    real_domain = real_domain_dir.name

                    if self.real_domains is not None and real_domain not in self.real_domains:
                        continue

                    for class_dir in real_domain_dir.iterdir():
                        if not class_dir.is_dir():
                            continue

                        class_name = class_dir.name

                        if class_name not in self.class_to_idx:
                            continue

                        label = self.class_to_idx[class_name]

                        for img_path in class_dir.rglob("*"):
                            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                                self.samples.append({
                                    "image_path": img_path,
                                    "label": label,
                                    "class_name": class_name,
                                    "domain_group": domain_group,
                                    "real_domain": real_domain,
                                    "split": split,
                                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = sample["label"]
        return image, label

In [ ]:
def create_model(model_path: str, num_classes: int, pretrained: bool = False):
    """
    Create a normal ERM classification model.

    - pretrained=False: train the whole model from scratch.
    - Keep pretrained=False for the paper-comparison from-scratch MobileNetV4 run.
    - All trainable parameters are optimized.
    """
    if model_path is None:
        raise ValueError("model_path must be specified.")
    if num_classes is None:
        raise ValueError("num_classes must be specified.")

    model = timm.create_model(
        model_path,
        pretrained=pretrained,
        num_classes=num_classes,
    )

    # Robust classifier reset for timm backbones.
    if hasattr(model, "reset_classifier"):
        model.reset_classifier(num_classes=num_classes)
    elif hasattr(model, "classifier") and isinstance(model.classifier, nn.Linear):
        model.classifier = nn.Linear(
            in_features=model.classifier.in_features,
            out_features=num_classes,
            bias=True,
        )
    elif hasattr(model, "fc") and isinstance(model.fc, nn.Linear):
        model.fc = nn.Linear(
            in_features=model.fc.in_features,
            out_features=num_classes,
            bias=True,
        )
    else:
        raise ValueError(
            "Could not find/reset classifier head. "
            "Check the timm model head name manually."
        )

    return model


In [ ]:
class MovingAverage:
    """
    DomainBed-style exponential moving average used by Fishr.

    This follows DomainBed-v2's MovingAverage behavior:
    - stores a detached EMA buffer per tensor name
    - returns ema_data / (1 - ema), so the gradient amplitude is not scaled by (1 - ema)
    """
    def __init__(self, ema: float, oneminusema_correction: bool = True):
        self.ema = ema
        self.ema_data = {}
        self._updates = 0
        self._oneminusema_correction = oneminusema_correction

    def update(self, dict_data):
        ema_dict_data = {}

        for name, data in dict_data.items():
            data = data.view(1, -1)

            if self._updates == 0:
                previous_data = torch.zeros_like(data)
            else:
                previous_data = self.ema_data[name].to(data.device)

            ema_data = self.ema * previous_data + (1.0 - self.ema) * data

            if self._oneminusema_correction:
                ema_dict_data[name] = ema_data / (1.0 - self.ema)
            else:
                ema_dict_data[name] = ema_data

            self.ema_data[name] = ema_data.detach().clone()

        self._updates += 1
        return ema_dict_data


def l2_between_dicts(dict_1, dict_2):
    assert len(dict_1) == len(dict_2)
    dict_1_values = [dict_1[key] for key in sorted(dict_1.keys())]
    dict_2_values = [dict_2[key] for key in sorted(dict_1.keys())]

    return (
        torch.cat([t.reshape(-1) for t in dict_1_values])
        - torch.cat([t.reshape(-1) for t in dict_2_values])
    ).pow(2).mean()


class TimmClassifierWithFeatures(nn.Module):
    """
    Small wrapper around a timm classifier.

    Normal forward returns logits exactly like the timm model.
    Fishr forward returns both:
    - logits
    - features immediately before the final linear classifier

    This lets us compute the Fishr gradient-variance penalty on the classifier
    without changing the backbone or adding MoCO / contrastive training.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def _get_linear_classifier(self):
        classifier = None

        if hasattr(self.model, "get_classifier"):
            classifier = self.model.get_classifier()

        candidate_modules = []

        if classifier is not None:
            if isinstance(classifier, nn.Module):
                candidate_modules.append(classifier)
            elif isinstance(classifier, (list, tuple)):
                candidate_modules.extend([m for m in classifier if isinstance(m, nn.Module)])

        for attr_name in ["classifier", "fc", "head"]:
            if hasattr(self.model, attr_name):
                module = getattr(self.model, attr_name)
                if isinstance(module, nn.Module):
                    candidate_modules.append(module)

        for module in candidate_modules:
            if isinstance(module, nn.Linear):
                return module

            linear_layers = [m for m in module.modules() if isinstance(m, nn.Linear)]
            if len(linear_layers) > 0:
                return linear_layers[-1]

        raise ValueError(
            "Fishr in this notebook expects a final nn.Linear classifier. "
            "For timm MobileNetV4 this should be available through model.get_classifier()."
        )

    def forward_logits_features(self, x):
        linear_classifier = self._get_linear_classifier()

        # Preferred timm path: get backbone features, apply head up to pre-logits,
        # then apply the final linear classifier manually.
        if hasattr(self.model, "forward_features") and hasattr(self.model, "forward_head"):
            features = self.model.forward_features(x)
            features = self.model.forward_head(features, pre_logits=True)

            if isinstance(features, (tuple, list)):
                features = features[0]

            if features.ndim > 2:
                features = torch.flatten(features, 1)

            logits = linear_classifier(features)
            return logits, features

        # Fallback path for non-timm models: capture the input to the final classifier.
        captured = {}

        def _capture_classifier_input(module, inputs):
            captured["features"] = inputs[0]

        handle = linear_classifier.register_forward_pre_hook(_capture_classifier_input)

        try:
            logits = self.model(x)
        finally:
            handle.remove()

        if "features" not in captured:
            raise RuntimeError("Could not capture features before the final classifier.")

        features = captured["features"]

        if isinstance(features, (tuple, list)):
            features = features[0]

        if features.ndim > 2:
            features = torch.flatten(features, 1)

        return logits, features

    def forward(self, x, return_features: bool = False):
        if return_features:
            return self.forward_logits_features(x)
        return self.model(x)

    def state_dict(self, *args, **kwargs):
        # Save the underlying timm model keys, so checkpoints stay compatible
        # with the original ERM notebook style.
        return self.model.state_dict(*args, **kwargs)

    def load_state_dict(self, state_dict, strict: bool = True):
        # Accept both wrapper-style keys ("model.xxx") and raw timm keys ("xxx").
        if len(state_dict) > 0 and all(k.startswith("model.") for k in state_dict.keys()):
            try:
                return super().load_state_dict(state_dict, strict=strict)
            except RuntimeError:
                stripped = {k[len("model."):]: v for k, v in state_dict.items()}
                return self.model.load_state_dict(stripped, strict=strict)

        return self.model.load_state_dict(state_dict, strict=strict)


## Fishr implementation notes

This notebook now keeps the original DomainBed-style pipeline, but changes the training update from ERM to normal Fishr. It still samples one minibatch from each source real domain, selects checkpoints only by source-domain validation, and evaluates target domains only after selection. No MoCO or contrastive objective is added.

The Fishr penalty is computed on the final linear classifier using the closed-form per-sample gradients of cross-entropy. This matches the DomainBed-v2 idea of matching per-domain classifier-gradient variances, without requiring `backpack-for-pytorch` in Kaggle.


In [ ]:
class Trainer:
    def __init__(
        self,
        model_path: str = None,
        num_classes: int = 60,
        pretrained: bool = False,
        seed: int = 42,
        root_path: str = "/kaggle/input/datasets/trieuvuongnguyen/nicopp-rethink-splitted/NICO++",
        target_domain=None,
        source_transform=None,
        target_transform=None,
        batch_size: int = 32,
        lr: float = 1e-4,
        weight_decay: float = 1e-4,
        max_iters: int = 10000,
        val_ratio: float = 0.2,
        source_splits=("train",),
        target_splits=("test",),
        num_workers: int = 2,
        use_amp: bool = True,
        use_data_parallel: bool = True,
        algorithm: str = "Fishr",
        fishr_lambda: float = 1000.0,
        fishr_penalty_anneal_iters: int = 1500,
        fishr_ema: float = 0.95,
    ):
        """
        DomainBed/DomainBed-v2 style trainer for ERM or normal Fishr.

        Important settings:
        - One minibatch is sampled from EACH source domain.
        - ERM concatenates the per-domain minibatches and optimizes cross entropy.
        - Fishr keeps the same ERM loss, plus a penalty that matches the
          per-domain variances of classifier gradients.
        - Model selection uses only held-out validation data from SOURCE domains.
        - Target domains are evaluated separately and are not used to select checkpoints.
        - No MoCO or contrastive objective is added here. Use pretrained=False for
          train-from-scratch with pretrained=False for this paper-comparison run.
        """

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.seed = seed
        set_seed(self.seed)

        self.root_path = root_path
        self.target_domain = target_domain

        self.model_path = model_path
        self.num_classes = num_classes
        self.pretrained = pretrained

        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.max_iters = max_iters

        self.val_ratio = val_ratio
        self.source_splits = tuple(source_splits)
        self.target_splits = tuple(target_splits)

        self.num_workers = num_workers
        self.use_data_parallel = use_data_parallel

        self.algorithm = algorithm.lower()
        if self.algorithm not in ["erm", "fishr"]:
            raise ValueError("algorithm must be either 'ERM' or 'Fishr'.")

        self.fishr_lambda = fishr_lambda
        self.fishr_penalty_anneal_iters = int(fishr_penalty_anneal_iters)
        self.fishr_ema = fishr_ema
        self.update_count = 0

        # Fishr needs stable differentiable gradient-variance computations.
        # AMP can make this penalty noisy, so keep Fishr in full precision.
        self.use_amp = use_amp and torch.cuda.is_available() and self.algorithm == "erm"
        if use_amp and self.algorithm == "fishr":
            print("Fishr selected: AMP is disabled for stable gradient-variance penalty computation.")

        self.scaler = GradScaler("cuda", enabled=self.use_amp)

        self.best_val_acc = -1.0
        self.best_state_dict = None
        self.history = []

        all_domain_groups = sorted([
            d.name for d in Path(root_path).iterdir()
            if d.is_dir()
        ])

        if target_domain is None:
            raise ValueError("target_domain must be specified.")

        if target_domain not in all_domain_groups:
            raise ValueError(
                f"target_domain '{target_domain}' not found. "
                f"Available domain groups: {all_domain_groups}"
            )

        self.source_domain_groups = [
            d for d in all_domain_groups
            if d != target_domain
        ]

        print(f"Algorithm: {self.algorithm.upper()}")
        print(f"Target group: {self.target_domain}")
        print(f"Source groups: {self.source_domain_groups}")
        print(f"Source splits used for train/val: {self.source_splits}")
        print(f"Target splits used for final eval: {self.target_splits}")

        # Build one global class mapping from class-folder names.
        # This does not load target images into training.
        class_index_dataset = NICOPPDataset(
            root_path=self.root_path,
            domain_groups=all_domain_groups,
            splits=("train", "test"),
            transform=None,
        )
        self.class_to_idx = class_index_dataset.class_to_idx

        if len(self.class_to_idx) != self.num_classes:
            print(
                f"Warning: num_classes={self.num_classes}, "
                f"but found {len(self.class_to_idx)} class folders. "
                f"Using found value."
            )
            self.num_classes = len(self.class_to_idx)

        print(f"Number of classes: {self.num_classes}")

        # Build model and wrap it so Fishr can access pre-classifier features.
        base_model = create_model(
            model_path=self.model_path,
            num_classes=self.num_classes,
            pretrained=self.pretrained,
        ).to(self.device)

        model = TimmClassifierWithFeatures(base_model).to(self.device)

        if self.algorithm == "fishr":
            # Fail early if this model has no final Linear classifier.
            final_classifier = model._get_linear_classifier()
            print(
                "Fishr classifier-gradient penalty will use final layer: "
                f"{final_classifier.__class__.__name__}"
            )

        if self.use_data_parallel and torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
            model = nn.DataParallel(model)
        else:
            print(f"Using device: {self.device}")

        self.model = model

        # Build one train loader per real source domain.
        # This mirrors DomainBed-v2: minibatches remain domain-separated until update().
        self.source_train_loaders = []
        self.source_val_subsets = []
        self.source_domain_names = []

        domain_counter = 0

        for domain_group in self.source_domain_groups:
            real_domains = self._get_real_domains(domain_group, self.source_splits)

            for real_domain in real_domains:
                train_dataset_full = NICOPPDataset(
                    root_path=self.root_path,
                    domain_groups=[domain_group],
                    splits=self.source_splits,
                    real_domains=[real_domain],
                    class_to_idx=self.class_to_idx,
                    transform=source_transform,
                )

                val_dataset_full = NICOPPDataset(
                    root_path=self.root_path,
                    domain_groups=[domain_group],
                    splits=self.source_splits,
                    real_domains=[real_domain],
                    class_to_idx=self.class_to_idx,
                    transform=target_transform,
                )

                n = len(train_dataset_full)

                if n == 0:
                    raise ValueError(
                        f"Empty source domain: group={domain_group}, real_domain={real_domain}"
                    )

                val_size = int(n * self.val_ratio)
                val_size = max(1, val_size)
                train_size = n - val_size

                if train_size <= 0:
                    raise ValueError(
                        f"Not enough samples for source domain: "
                        f"group={domain_group}, real_domain={real_domain}, n={n}"
                    )

                generator = torch.Generator().manual_seed(self.seed + domain_counter)
                indices = torch.randperm(n, generator=generator).tolist()

                train_indices = indices[:train_size]
                val_indices = indices[train_size:]

                train_subset = Subset(train_dataset_full, train_indices)
                val_subset = Subset(val_dataset_full, val_indices)

                # DomainBed-v2 uses one per-domain batch size.
                # If a tiny domain has fewer samples than batch_size, do not drop its only batch.
                drop_last = len(train_subset) >= self.batch_size

                loader_kwargs = dict(
                    batch_size=self.batch_size,
                    shuffle=True,
                    num_workers=self.num_workers,
                    pin_memory=torch.cuda.is_available(),
                    drop_last=drop_last,
                )

                if self.num_workers > 0:
                    loader_kwargs.update(
                        persistent_workers=True,
                        prefetch_factor=2,
                    )

                train_loader = DataLoader(train_subset, **loader_kwargs)

                if len(train_loader) == 0:
                    raise ValueError(
                        f"Zero batches for source domain {domain_group}/{real_domain}. "
                        f"Reduce batch_size."
                    )

                self.source_train_loaders.append(train_loader)
                self.source_val_subsets.append(val_subset)
                self.source_domain_names.append(f"{domain_group}/{real_domain}")

                print(
                    f"Source real domain: {domain_group}/{real_domain:8s} "
                    f"| train={len(train_subset)} "
                    f"| val={len(val_subset)} "
                    f"| batches={len(train_loader)}"
                )

                domain_counter += 1

        if len(self.source_train_loaders) == 0:
            raise ValueError("No source loaders were created.")

        self.num_source_domains = len(self.source_train_loaders)

        if self.algorithm == "fishr":
            self.fishr_ema_per_domain = [
                MovingAverage(ema=self.fishr_ema, oneminusema_correction=True)
                for _ in range(self.num_source_domains)
            ]

            print(
                f"Fishr hparams | lambda={self.fishr_lambda} "
                f"| penalty_anneal_iters={self.fishr_penalty_anneal_iters} "
                f"| ema={self.fishr_ema}"
            )

        self.source_val_dataset = ConcatDataset(self.source_val_subsets)

        if len(self.source_val_dataset) == 0:
            raise ValueError(
                "Source validation dataset is empty. "
                "Increase val_ratio or check your source datasets."
            )

        eval_batch_size = self.batch_size * len(self.source_train_loaders)

        self.source_val_loader = DataLoader(
            self.source_val_dataset,
            batch_size=eval_batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

        print(f"Source val samples: {len(self.source_val_dataset)}")

        # Target test loaders: one loader per real target domain.
        self.target_real_domains = self._get_real_domains(target_domain, self.target_splits)

        if len(self.target_real_domains) == 0:
            raise ValueError(
                f"No real target domains found in target group={target_domain}. "
                f"Check folder structure and target_splits={self.target_splits}."
            )

        self.target_datasets = {}
        self.target_loaders = {}

        for real_domain in self.target_real_domains:
            target_dataset = NICOPPDataset(
                root_path=self.root_path,
                domain_groups=[target_domain],
                splits=self.target_splits,
                real_domains=[real_domain],
                class_to_idx=self.class_to_idx,
                transform=target_transform,
            )

            if len(target_dataset) == 0:
                raise ValueError(
                    f"Empty target domain: group={target_domain}, real_domain={real_domain}"
                )

            target_loader = DataLoader(
                target_dataset,
                batch_size=eval_batch_size,
                shuffle=False,
                num_workers=self.num_workers,
                pin_memory=torch.cuda.is_available(),
            )

            self.target_datasets[real_domain] = target_dataset
            self.target_loaders[real_domain] = target_loader

            print(
                f"Target real domain: {real_domain:8s} "
                f"| samples={len(target_dataset)}"
            )

        self.target_dataset = ConcatDataset(list(self.target_datasets.values()))

        self.target_loader = DataLoader(
            self.target_dataset,
            batch_size=eval_batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

        print(f"Target group samples: {len(self.target_dataset)}")
        print(f"Batch size per source domain: {self.batch_size}")
        print(f"Effective train batch size: {self.batch_size * len(self.source_train_loaders)}")

        self.criterion = nn.CrossEntropyLoss()

        # DomainBed-v2 computes steps_per_epoch from the smallest source domain.
        # Use ceil(dataset_size / batch_size), not len(loader), to match its train.py logic.
        self.steps_per_epoch = math.ceil(
            min(len(loader.dataset) for loader in self.source_train_loaders) / self.batch_size
        )

        if self.steps_per_epoch <= 0:
            raise ValueError("At least one source train loader has zero batches. Try reducing batch_size.")

        self.max_epochs = math.ceil(self.max_iters / self.steps_per_epoch)

        self._init_optimizer_and_scheduler()

        print(f"Steps per epoch: {self.steps_per_epoch}")
        print(f"Max epochs for scheduler: {self.max_epochs}")

    def _init_optimizer_and_scheduler(self):
        self.optimizer = Adam(
            self.model.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )

        self.scheduler = CosineAnnealingLR(
            self.optimizer,
            T_max=self.max_epochs,
        )

    def _get_real_domains(self, domain_group, splits):
        real_domains = set()

        for split in splits:
            split_path = Path(self.root_path) / domain_group / split
            if not split_path.exists():
                continue

            for real_domain_dir in split_path.iterdir():
                if real_domain_dir.is_dir():
                    real_domains.add(real_domain_dir.name)

        return sorted(real_domains)

    def get_base_model(self):
        if isinstance(self.model, nn.DataParallel):
            return self.model.module
        return self.model

    def _set_train_mode(self):
        # Normal full training: keep the whole model in train mode.
        self.model.train()

    def evaluate(self, loader):
        self.model.eval()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        with torch.no_grad():
            for images, labels in loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True)

                with autocast("cuda", enabled=self.use_amp):
                    logits = self.model(images)
                    loss = self.criterion(logits, labels)

                preds = logits.argmax(dim=1)

                cur_batch_size = labels.size(0)
                total_loss += loss.item() * cur_batch_size
                total_correct += (preds == labels).sum().item()
                total_samples += cur_batch_size

        avg_loss = total_loss / total_samples
        avg_acc = total_correct / total_samples

        return {
            "loss": avg_loss,
            "acc": avg_acc,
            "total_correct": total_correct,
            "total": total_samples,
        }

    def _next_source_minibatches(self, source_iters):
        minibatches = []

        for i, loader in enumerate(self.source_train_loaders):
            try:
                images, labels = next(source_iters[i])
            except StopIteration:
                source_iters[i] = iter(loader)
                images, labels = next(source_iters[i])

            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            minibatches.append((images, labels))

        return minibatches

    def _forward_logits_features(self, x):
        return self.model(x, return_features=True)

    def _erm_update(self, minibatches):
        # Same ERM logic as DomainBed-v2:
        # all_x = cat([x for x, y in minibatches])
        # all_y = cat([y for x, y in minibatches])
        all_x = torch.cat([x for x, y in minibatches], dim=0)
        all_y = torch.cat([y for x, y in minibatches], dim=0)

        with autocast("cuda", enabled=self.use_amp):
            logits = self.model(all_x)
            loss = self.criterion(logits, all_y)

        self.optimizer.zero_grad(set_to_none=True)
        self.scaler.scale(loss).backward()
        self.scaler.step(self.optimizer)
        self.scaler.update()

        preds = logits.argmax(dim=1)
        train_acc = (preds == all_y).float().mean().item()

        return {
            "loss": loss.item(),
            "nll": loss.item(),
            "penalty": 0.0,
            "penalty_weight": 0.0,
            "train_acc": train_acc,
            "seen_samples": all_y.size(0),
        }

    def _compute_fishr_penalty(self, logits, labels, features, len_minibatches):
        """
        Native Fishr penalty for a final Linear classifier.

        DomainBed-v2 computes per-sample gradients of the classifier with Backpack.
        For a Linear classifier, the same per-sample gradient is available in closed form:

            grad_W_i = (softmax(logits_i) - one_hot(y_i)) outer features_i
            grad_b_i =  softmax(logits_i) - one_hot(y_i)

        We then match the variance of these per-sample gradients across domains.
        """
        probs = F.softmax(logits, dim=1)
        one_hot = F.one_hot(labels, num_classes=logits.size(1)).to(dtype=logits.dtype)
        deltas = probs - one_hot

        features = features.float()
        deltas = deltas.float()

        grads_var_per_domain = []
        start = 0

        for bsize in len_minibatches:
            env_delta = deltas[start:start + bsize]
            env_features = features[start:start + bsize]
            start += bsize

            # Variance of classifier weight gradients:
            # Var(delta_c * feature_f) = E[delta_c^2 * feature_f^2] - E[delta_c * feature_f]^2
            mean_w = torch.einsum("bc,bf->cf", env_delta, env_features) / bsize
            mean_w_sq = torch.einsum("bc,bf->cf", env_delta.pow(2), env_features.pow(2)) / bsize
            var_w = (mean_w_sq - mean_w.pow(2)).clamp_min(0.0)

            # Variance of classifier bias gradients.
            mean_b = env_delta.mean(dim=0)
            mean_b_sq = env_delta.pow(2).mean(dim=0)
            var_b = (mean_b_sq - mean_b.pow(2)).clamp_min(0.0)

            grads_var_per_domain.append(OrderedDict(weight=var_w, bias=var_b))

        # DomainBed-style EMA per source domain.
        for domain_id in range(self.num_source_domains):
            grads_var_per_domain[domain_id] = self.fishr_ema_per_domain[domain_id].update(
                grads_var_per_domain[domain_id]
            )

        grads_var_mean = OrderedDict([
            (
                name,
                torch.stack([
                    grads_var_per_domain[domain_id][name]
                    for domain_id in range(self.num_source_domains)
                ], dim=0).mean(dim=0),
            )
            for name in grads_var_per_domain[0].keys()
        ])

        penalty = 0.0
        for domain_id in range(self.num_source_domains):
            penalty = penalty + l2_between_dicts(
                grads_var_per_domain[domain_id],
                grads_var_mean,
            )

        return penalty / self.num_source_domains

    def _fishr_update(self, minibatches):
        assert len(minibatches) == self.num_source_domains

        all_x = torch.cat([x for x, y in minibatches], dim=0)
        all_y = torch.cat([y for x, y in minibatches], dim=0)
        len_minibatches = [x.shape[0] for x, y in minibatches]

        logits, features = self._forward_logits_features(all_x)

        nll = F.cross_entropy(logits, all_y)
        penalty = self._compute_fishr_penalty(
            logits=logits,
            labels=all_y,
            features=features,
            len_minibatches=len_minibatches,
        )

        if self.update_count >= self.fishr_penalty_anneal_iters:
            penalty_weight = self.fishr_lambda
        else:
            penalty_weight = 0.0

        # Reset Adam at the annealing boundary, matching the DomainBed-v2 Fishr style.
        if self.update_count == self.fishr_penalty_anneal_iters and self.fishr_penalty_anneal_iters != 0:
            self._init_optimizer_and_scheduler()

        objective = nll + penalty_weight * penalty

        self.optimizer.zero_grad(set_to_none=True)
        objective.backward()
        self.optimizer.step()

        self.update_count += 1

        preds = logits.argmax(dim=1)
        train_acc = (preds == all_y).float().mean().item()

        return {
            "loss": objective.item(),
            "nll": nll.item(),
            "penalty": penalty.item(),
            "penalty_weight": float(penalty_weight),
            "train_acc": train_acc,
            "seen_samples": all_y.size(0),
        }

    def _update(self, minibatches):
        if self.algorithm == "fishr":
            return self._fishr_update(minibatches)
        return self._erm_update(minibatches)

    def train(
        self,
        eval_iters=600,
        log_iters=20,
        save_path=None,
        eval_target_during_training=False,
    ):
        self._set_train_mode()

        print(f"Training {self.algorithm.upper()} with {self.max_iters} iterations")
        print(f"Target group: {self.target_domain}")
        print(f"Source real domains: {self.source_domain_names}")
        print(f"Validation/checkpoint frequency: every {eval_iters} iterations")
        print(f"Logging frequency: every {log_iters} iterations")

        source_iters = [iter(loader) for loader in self.source_train_loaders]

        best_val_acc = -1.0
        best_state_dict = None
        history = []

        running_loss = 0.0
        running_nll = 0.0
        running_penalty = 0.0
        running_acc_sum = 0.0
        running_steps = 0
        last_update_stats = None

        pbar = tqdm(range(1, self.max_iters + 1), total=self.max_iters)

        for step in pbar:
            minibatches = self._next_source_minibatches(source_iters)
            update_stats = self._update(minibatches)
            last_update_stats = update_stats

            running_loss += update_stats["loss"]
            running_nll += update_stats["nll"]
            running_penalty += update_stats["penalty"]
            running_acc_sum += update_stats["train_acc"]
            running_steps += 1

            # DomainBed-v2 steps the scheduler once per epoch.
            if step % self.steps_per_epoch == 0:
                self.scheduler.step()

            if step % log_iters == 0:
                train_loss = running_loss / running_steps
                train_nll = running_nll / running_steps
                train_penalty = running_penalty / running_steps
                train_acc_avg = running_acc_sum / running_steps

                postfix = {
                    "loss": f"{train_loss:.4f}",
                    "train_acc": f"{train_acc_avg:.4f}",
                    "lr": f"{self.optimizer.param_groups[0]['lr']:.6f}",
                }

                if self.algorithm == "fishr":
                    postfix.update({
                        "nll": f"{train_nll:.4f}",
                        "fishr_penalty": f"{train_penalty:.4f}",
                        "penalty_w": f"{last_update_stats['penalty_weight']:.1f}",
                    })

                pbar.set_postfix(postfix)

                running_loss = 0.0
                running_nll = 0.0
                running_penalty = 0.0
                running_acc_sum = 0.0
                running_steps = 0

            if step % eval_iters == 0 or step == self.max_iters:
                source_val_metrics = self.evaluate(self.source_val_loader)

                record = {
                    "iter": step,
                    "algorithm": self.algorithm.upper(),
                    "source_val_loss": source_val_metrics["loss"],
                    "source_val_acc": source_val_metrics["acc"],
                    "lr": self.optimizer.param_groups[0]["lr"],
                }

                if last_update_stats is not None:
                    record["train_loss"] = last_update_stats["loss"]
                    record["train_nll"] = last_update_stats["nll"]
                    record["fishr_penalty"] = last_update_stats["penalty"]
                    record["fishr_penalty_weight"] = last_update_stats["penalty_weight"]

                print(
                    f"\nIter [{step}/{self.max_iters}] "
                    f"| Source Val Loss: {source_val_metrics['loss']:.4f} "
                    f"| Source Val Acc: {source_val_metrics['acc']:.4f} "
                    f"| LR: {self.optimizer.param_groups[0]['lr']:.6f}"
                )

                if self.algorithm == "fishr" and last_update_stats is not None:
                    print(
                        f"Fishr | NLL: {last_update_stats['nll']:.4f} "
                        f"| Penalty: {last_update_stats['penalty']:.6f} "
                        f"| Penalty Weight: {last_update_stats['penalty_weight']:.1f}"
                    )

                # DomainBed-v2 logs test performance at checkpoints, but DOES NOT
                # use it for model selection. Best model is selected only by source val.
                if eval_target_during_training:
                    target_metrics = self.evaluate_target(load_best=False, verbose=False)
                    record["target_weighted_acc"] = target_metrics["weighted_acc"]
                    record["target_macro_acc"] = target_metrics["macro_acc"]
                    record["target_domain_acc"] = {
                        k: v["acc"] for k, v in target_metrics["domain_metrics"].items()
                    }
                    print(
                        f"Target Weighted Acc: {target_metrics['weighted_acc']:.4f} "
                        f"| Target Macro Acc: {target_metrics['macro_acc']:.4f}"
                    )

                history.append(record)

                if source_val_metrics["acc"] > best_val_acc:
                    best_val_acc = source_val_metrics["acc"]
                    best_state_dict = {
                        k: v.detach().cpu().clone()
                        for k, v in self.get_base_model().state_dict().items()
                    }

                    print(f"New best source-val acc at iter {step}: {best_val_acc:.4f}")

                    if save_path is not None:
                        torch.save({
                            "iter": step,
                            "algorithm": self.algorithm.upper(),
                            "model_state_dict": best_state_dict,
                            "best_val_acc": best_val_acc,
                            "class_to_idx": self.class_to_idx,
                            "source_domain_groups": self.source_domain_groups,
                            "source_domain_names": self.source_domain_names,
                            "target_domain": self.target_domain,
                            "target_real_domains": self.target_real_domains,
                            "source_splits": self.source_splits,
                            "target_splits": self.target_splits,
                            "model_path": self.model_path,
                            "num_classes": self.num_classes,
                            "pretrained": self.pretrained,
                            "lr": self.lr,
                            "weight_decay": self.weight_decay,
                            "batch_size_per_domain": self.batch_size,
                            "max_iters": self.max_iters,
                            "eval_iters": eval_iters,
                            "seed": self.seed,
                            "fishr_lambda": self.fishr_lambda,
                            "fishr_penalty_anneal_iters": self.fishr_penalty_anneal_iters,
                            "fishr_ema": self.fishr_ema,
                        }, save_path)

                self._set_train_mode()

        self.best_val_acc = best_val_acc
        self.best_state_dict = best_state_dict
        self.history = history

        print(f"\nBest Source Val Acc: {best_val_acc:.4f}")

        return history

    def evaluate_target(self, load_best=True, verbose=True):
        if load_best and self.best_state_dict is not None:
            self.get_base_model().load_state_dict(self.best_state_dict)

        self.model.eval()

        domain_metrics = {}

        for real_domain, loader in self.target_loaders.items():
            metrics = self.evaluate(loader)
            domain_metrics[real_domain] = metrics

        # Macro average across target domains: useful for paper-style domain columns.
        macro_acc = sum(m["acc"] for m in domain_metrics.values()) / len(domain_metrics)
        macro_loss = sum(m["loss"] for m in domain_metrics.values()) / len(domain_metrics)

        # Weighted average across target samples: this matches DomainBed-v2 "overall".
        total_correct = sum(m["total_correct"] for m in domain_metrics.values())
        total_samples = sum(m["total"] for m in domain_metrics.values())

        weighted_acc = total_correct / total_samples
        weighted_loss = sum(
            m["loss"] * m["total"] for m in domain_metrics.values()
        ) / total_samples

        if verbose:
            print("\nTarget Evaluation")
            print(f"Target group: {self.target_domain}")

            for real_domain, metrics in domain_metrics.items():
                print(
                    f"{real_domain:8s} "
                    f"| Loss: {metrics['loss']:.4f} "
                    f"| Acc: {metrics['acc']:.4f} "
                    f"| Correct: {metrics['total_correct']} / {metrics['total']}"
                )

            print(f"Macro Acc across target domains: {macro_acc:.4f}")
            print(f"Weighted Acc across target samples / DomainBed-v2 overall: {weighted_acc:.4f}")

        return {
            "target_group": self.target_domain,
            "domain_metrics": domain_metrics,
            "macro_loss": macro_loss,
            "macro_acc": macro_acc,
            "weighted_loss": weighted_loss,
            "weighted_acc": weighted_acc,
            "total_correct": total_correct,
            "total": total_samples,
        }


In [ ]:
source_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.3),
    transforms.RandomGrayscale(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

target_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
trainer = Trainer(
    model_path="mobilenetv4_conv_small.e2400_r224_in1k",
    num_classes=60,
    pretrained=False,                  # normal Fishr from scratch; no MoCO / contrastive pretraining
    algorithm="Fishr",
    fishr_lambda=1000.0,               # DomainBed-v2 default for Fishr
    fishr_penalty_anneal_iters=1500,   # DomainBed-v2 default annealing point
    fishr_ema=0.95,                    # DomainBed-v2 default EMA
    target_domain="autumn_rock",       # NICO++ leave-one-group-out: autumn + rock are target
    source_transform=source_transform,
    target_transform=target_transform,
    batch_size=32,                     # per-source-domain batch size
    lr=1e-4,
    weight_decay=1e-4,
    max_iters=10000,                    # DomainBed-v2 NICO++ setting from the paper
    val_ratio=0.2,                     # source-domain validation holdout
    source_splits=("train",),          # source train is internally split into train/val
    target_splits=("test",),           # target is held out for final evaluation
    num_workers=2,
    use_amp=True,                      # Fishr disables AMP internally for penalty stability
    use_data_parallel=True,            # works with DataParallel; set False if memory is tight
)


In [ ]:
history = trainer.train(
    eval_iters=600,
    log_iters=100,
    save_path="/kaggle/working/best_fishr.pth",
    eval_target_during_training=False,  # keep target test unseen during training/checkpoint selection
)


In [ ]:
NICO_PP_TARGET_GROUPS = ["autumn_rock", "dim_grass", "outdoor_water"]
NICO_PP_PAPER_DOMAIN_ORDER = ["autumn", "rock", "dim", "grass", "outdoor", "water"]


def paper_metric_row(results_by_group, algorithm_name=None):
    """
    Return NICO++ paper-style columns: each real target-domain accuracy in percent
    plus the macro Avg across available target domains.

    Pass either one evaluate_target() result or a dict such as:
        {"autumn_rock": autumn_rock_results, "dim_grass": dim_grass_results, ...}
    """
    if isinstance(results_by_group, dict) and (
        "domain_metrics" in results_by_group or "per_domain" in results_by_group
    ):
        grouped_results = {results_by_group.get("target_group", "target"): results_by_group}
    else:
        grouped_results = results_by_group

    accs = {}
    for _, result in grouped_results.items():
        if isinstance(result, dict) and "target_results" in result:
            result = result["target_results"]

        domain_metrics = result.get("domain_metrics", result.get("per_domain", {}))
        for domain, metrics in domain_metrics.items():
            real_domain = str(domain).split("/")[-1]
            acc = float(metrics["acc"])
            accs[real_domain] = acc * 100.0 if acc <= 1.0 else acc

    row = {domain: accs.get(domain, np.nan) for domain in NICO_PP_PAPER_DOMAIN_ORDER}
    values = list(row.values())
    row["Avg"] = float(np.nanmean(values)) if np.isfinite(values).any() else np.nan

    if algorithm_name is not None:
        row = {"Algorithm": algorithm_name, **row}

    return pd.DataFrame([row])


In [ ]:
ckpt_path = "/kaggle/working/best_fishr.pth"

checkpoint = torch.load(ckpt_path, map_location="cpu")
state_dict = checkpoint["model_state_dict"]

trainer.get_base_model().load_state_dict(state_dict)

target_results = trainer.evaluate_target()

paper_metrics = paper_metric_row(target_results, algorithm_name="Fishr")
paper_metrics
